In [15]:
from diffwave.inference import predict as diffwave_predict
import os
import numpy as np
model_dir = 'diffwav_model/diffwave-ljspeech-22kHz-1000578.pt'
from diffwave.inference import *
from diffwave.preprocess import *
# audio is a GPU tensor in [N,T] format.

In [16]:
wm_path = './audioseal_squarewav_50'
sound_file = os.listdir(wm_path)
save_pth = 'audioseal_squarewav_diffrecover_50'
state = 'diff'
for i in sound_file:
    spectrogram = transform(f'{wm_path}/{i}')
    audio, sample_rate = diffwave_predict(torch.tensor(spectrogram,dtype=torch.float32).unsqueeze(0).cuda(), model_dir = model_dir, fast_sampling=True)
    torchaudio.save(f"./{save_pth}/{state}_{i.split('.')[0]}.wav", audio.cpu(), sample_rate)


/tmp/ipykernel_2714979/2632586405.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  audio, sample_rate = diffwave_predict(torch.tensor(spectrogram,dtype=torch.float32).unsqueeze(0).cuda(), model_dir = model_dir, fast_sampling=True)


# Test Model

In [17]:
import torch
import torchaudio
import urllib

import sys
sys.path.append("..")  # 添加上一级目录
from src.audioseal import AudioSeal

model = AudioSeal.load_generator("audioseal_wm_16bits")

In [19]:
import os
secret_dir = './my_secret_50'
acc = []
detector = AudioSeal.load_detector(("audioseal_detector_16bits"))
data_path = save_pth
my_seal_audio = os.listdir(data_path)

for i in my_seal_audio:
    wav, sr = torchaudio.load(f'{data_path}/{i}')

    secret_mesage = torch.load(f'./{secret_dir}/{i.split("_")[-1]}_msg_tensor.pt')
    result, message = detector.detect_watermark(wav.unsqueeze(0), sample_rate=sr, message_threshold=0.5)
    acc.append(result)

acc

/tmp/ipykernel_2714979/2377774728.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  secret_mesage = torch.load(f'./{secret_dir}/{i.split("_")[-1]}_msg_tensor.pt')


[0.010975729674100876,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0036199011374264956,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0003532062692102045,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.004550362471491098,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.004503017757087946,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0]

In [20]:
sum(acc)/len(acc)

0.0004800443461863324